# cthreads CPU demo: Mandelbrot

This notebook compares a **pure Python** Mandelbrot renderer with the same algorithm
compiled by **cthreads** (`@Thread`). You should see a large speedup after the first
compile, plus matching images.

## What you need

1. `pip install cthreads` (CPU wheel from PyPI)
2. A C++ compiler for the first kernel build (Colab: `g++` via `apt`)

Runtime note: the **first** `thread(...)` pays codegen + compile. We warm up once,
then time steady-state runs so the comparison is fair.

## 1. Setup

In [ ]:
%pip install -q cthreads matplotlib
!apt-get -qq install -y g++

## 2. Shared parameters

We render into a flat `list[int]` of length `width * height` (iteration count per pixel).

In [ ]:
WIDTH: int = 640
HEIGHT: int = 480
MAX_ITER: int = 80
X_MIN: float = -2.0
X_MAX: float = 1.0
Y_MIN: float = -1.2
Y_MAX: float = 1.2

N: int = WIDTH * HEIGHT

## 3. Pure Python baseline

Same math as the cthreads kernel below — nested loops, escape time, write `out[i]`.

In [ ]:
def mandelbrot_python(
    width: int,
    height: int,
    max_iter: int,
    x_min: float,
    x_max: float,
    y_min: float,
    y_max: float,
    out: list[int],
) -> None:
    for py in range(height):
        for px in range(width):
            x0: float = x_min + (x_max - x_min) * (1.0 * px) / (1.0 * width)
            y0: float = y_min + (y_max - y_min) * (1.0 * py) / (1.0 * height)
            x: float = 0.0
            y: float = 0.0
            i: int = 0
            while i < max_iter:
                if x * x + y * y > 4.0:
                    break
                xt: float = x * x - y * y + x0
                y = 2.0 * x * y + y0
                x = xt
                i = i + 1
            out[py * width + px] = i

## 4. Same algorithm as a `@Thread` kernel

`cthreads.thread(...)` runs a **compiled C++** copy off the GIL. Locals need type
annotations; results come back via list writeback on `join()`.

In [ ]:
import cthreads
from cthreads import Thread

@Thread
def mandelbrot_native(
    width: int,
    height: int,
    max_iter: int,
    x_min: float,
    x_max: float,
    y_min: float,
    y_max: float,
    out: list[int],
) -> None:
    for py in range(height):
        for px in range(width):
            x0: float = x_min + (x_max - x_min) * (1.0 * px) / (1.0 * width)
            y0: float = y_min + (y_max - y_min) * (1.0 * py) / (1.0 * height)
            x: float = 0.0
            y: float = 0.0
            i: int = 0
            while i < max_iter:
                if x * x + y * y > 4.0:
                    break
                xt: float = x * x - y * y + x0
                y = 2.0 * x * y + y0
                x = xt
                i = i + 1
            out[py * width + px] = i

## 5. Warmup, then time both

Warmup compiles the kernel once. Timed runs measure steady-state performance.

In [ ]:
import time

out_py: list[int] = [0] * N
out_native: list[int] = [0] * N

# --- warmup (do not time) ---
cthreads.thread(
    mandelbrot_native,
    WIDTH, HEIGHT, MAX_ITER, X_MIN, X_MAX, Y_MIN, Y_MAX, out_native,
).join()

# --- timed pure Python ---
out_py = [0] * N
t0 = time.perf_counter()
mandelbrot_python(WIDTH, HEIGHT, MAX_ITER, X_MIN, X_MAX, Y_MIN, Y_MAX, out_py)
t_python = time.perf_counter() - t0

# --- timed cthreads ---
out_native = [0] * N
t0 = time.perf_counter()
cthreads.thread(
    mandelbrot_native,
    WIDTH, HEIGHT, MAX_ITER, X_MIN, X_MAX, Y_MIN, Y_MAX, out_native,
).join()
t_native = time.perf_counter() - t0

print(f"pure Python:  {t_python * 1000:.1f} ms")
print(f"cthreads:     {t_native * 1000:.1f} ms")
if t_native > 0.0:
    print(f"speedup:      {t_python / t_native:.1f}x")

mismatches = sum(1 for a, b in zip(out_py, out_native) if a != b)
print(f"pixel mismatches: {mismatches}")

## 6. Visual check

Left / right: Python vs cthreads images. Bottom: timing bars.

In [ ]:
import matplotlib.pyplot as plt

img_py = [out_py[r * WIDTH:(r + 1) * WIDTH] for r in range(HEIGHT)]
img_native = [out_native[r * WIDTH:(r + 1) * WIDTH] for r in range(HEIGHT)]

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].imshow(img_py, cmap="magma", origin="lower")
axes[0].set_title("Pure Python")
axes[0].axis("off")

axes[1].imshow(img_native, cmap="magma", origin="lower")
axes[1].set_title("cthreads @Thread")
axes[1].axis("off")

axes[2].bar(["Pure Python", "cthreads"], [t_python * 1000.0, t_native * 1000.0], color=["#888888", "#2a9d8f"])
axes[2].set_ylabel("ms")
axes[2].set_title("Render time")

fig.tight_layout()
plt.show()

## Takeaways

- Annotate parameters and locals; use allowed types (`int`, `float`, `list[...]`, …).
- Launch with `cthreads.thread(fn, *args).join()`; mutable lists are written back on join.
- First launch compiles; later launches reuse the cache.

More detail: [docs/concepts.md](https://github.com/K-T0BIAS/CThreads/blob/main/docs/concepts.md),
[docs/quickstart](https://github.com/K-T0BIAS/CThreads/blob/main/docs/quickstart.md).